# CMMD2022 Veri Kesfi — Parca 1

`PLAN.md` Bolum 1'deki varsayim tablosunu dogrulamak icin. Bu notebook **hicbir sey degistirmez**,
sadece olcer ve `ozet/` altina yazar.

Hem **yerelde** (`archive/TheChineseMammographyDatabase`) hem **Colab**'da (Drive) calisir,
ortami kendi tespit eder.

Olculecekler: dosya envanteri, klinik XLSX kolonlari/degerleri, goruntu cozunurlugu, goruntuleme
yonu (CC/MLO), `PhotometricInterpretation`, bit derinligi, meme basina goruntu sayisi,
klinik etiketle eslesme, benign/malign orani, TCIA'nin uyardigi yinelenen piksel hash'leri.

In [ ]:
import os, glob, hashlib, json, collections
from pathlib import Path

ORTAM = 'colab' if os.path.isdir('/content') else 'yerel'
print('ortam:', ORTAM)

if ORTAM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

# Veri kokunu otomatik bul: TCIA klasoru (icinde CMMD/ ve CMMD_clinicaldata_revision.xlsx var)
ADAY_KOKLER = [
    r'D:\mamografi\multiple_instance_classifier\archive\TheChineseMammographyDatabase',
    '/content/drive/MyDrive/multiple_instance_classifier/archive/TheChineseMammographyDatabase',
    '/content/drive/MyDrive/multiple_instance_classifier/archive',
    '/content/cmmd2022',
]
DATA_ROOT = next((p for p in ADAY_KOKLER if os.path.isdir(p)), None)
print('DATA_ROOT:', DATA_ROOT)
if DATA_ROOT is None:
    print('Veri bulunamadi -> asagidaki Kaggle indirme hucresini calistir veya ADAY_KOKLER a yol ekle.')

### Kaggle'dan indirme (yalnizca veri bulunamadiysa)

`kaggle.json` Drive'da `MyDrive/kaggle.json` olarak duruyorsa calisir. Veri bulunduysa bu hucre
kendini atlar.

In [ ]:
if DATA_ROOT is None and ORTAM == 'colab':
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp /content/drive/MyDrive/kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d tommyngx/cmmd2022 -p /content/cmmd2022 --unzip
    aday = glob.glob('/content/cmmd2022/**/CMMD_clinicaldata_revision.xlsx', recursive=True)
    DATA_ROOT = os.path.dirname(aday[0]) if aday else '/content/cmmd2022'
    print('DATA_ROOT:', DATA_ROOT)

In [ ]:
try:
    import pydicom
except ImportError:
    !pip -q install pydicom
    import pydicom

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ozet ciktilari: Colab'da Drive'a, yerelde repo icine
OZET_DIR = ('/content/drive/MyDrive/multiple_instance_classifier/ozet' if ORTAM == 'colab' else str(Path.cwd() / 'ozet'))
os.makedirs(OZET_DIR, exist_ok=True)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
print('ozet cikti:', OZET_DIR)

## 1. Dosya envanteri

TCIA hiyerarsisi: `CMMD/<hasta>/<tetkik>/<seri>/1-N.dcm`.

In [ ]:
tum = [p for p in glob.glob(os.path.join(DATA_ROOT, '**', '*'), recursive=True) if os.path.isfile(p)]
uzanti = collections.Counter(Path(p).suffix.lower() for p in tum)

print(f'toplam dosya: {len(tum)}')
for u, n in uzanti.most_common(10):
    print(f'  {u or "(uzantisiz)":15s} {n}')

dicom_dosyalar = sorted(p for p in tum if Path(p).suffix.lower() == '.dcm')
tablo_dosyalar = [p for p in tum if Path(p).suffix.lower() in {'.xlsx', '.csv'}]
print(f'\ndicom: {len(dicom_dosyalar)}')
print('tablo dosyalari:', [os.path.relpath(p, DATA_ROOT) for p in tablo_dosyalar])
for p in dicom_dosyalar[:3]:
    print(' ornek yol:', os.path.relpath(p, DATA_ROOT))

## 2. Klinik veri

Beklenen kolonlar: `ID1`, `LeftRight`, `Age`, `number`, `abnormality`, `classification`, `subtype`.

In [ ]:
KLINIK_YOL = next((p for p in tablo_dosyalar if 'clinical' in os.path.basename(p).lower()), None)
if KLINIK_YOL is None:
    raise SystemExit(f'Klinik tablo bulunamadi: {tablo_dosyalar}')

klinik = (pd.read_excel(KLINIK_YOL) if KLINIK_YOL.lower().endswith('.xlsx')
          else pd.read_csv(KLINIK_YOL))
print('klinik dosya:', os.path.relpath(KLINIK_YOL, DATA_ROOT), klinik.shape)

print('\nkolonlar / tipler:')
print(klinik.dtypes.to_string())
print('\neksik deger:')
print(klinik.isna().sum().to_string())
print('\nilk 5 satir:')
print(klinik.head().to_string(index=False))

print('\nbenzersiz degerler:')
for k in klinik.columns:
    u = klinik[k].dropna().unique()
    print(f'  {k}: {len(u)} benzersiz', sorted(map(str, u)) if len(u) <= 12 else list(map(str, u[:6])))

## 3. DICOM meta taramasi

Onemli: bu veride `ViewPosition` (0018,5101) **bos**. Goruntuleme yonu yalnizca
`ViewCodeSequence[0].CodeMeaning` icinde (`cranio-caudal` / `medio-lateral oblique`).
Pikseller okunmuyor, sadece header — hizli.

In [ ]:
CC_MLO = {'cranio-caudal': 'CC', 'medio-lateral oblique': 'MLO'}


def dicom_meta(yol):
    d = pydicom.dcmread(yol, stop_before_pixels=True, force=True)
    vcs = getattr(d, 'ViewCodeSequence', None)
    kod = str(getattr(vcs[0], 'CodeMeaning', '')) if vcs else ''
    g = lambda k: str(getattr(d, k, '')) or None
    return dict(
        yol=os.path.relpath(yol, DATA_ROOT),
        hasta=g('PatientID'),
        taraf=g('ImageLaterality') or g('Laterality'),
        view=CC_MLO.get(kod, kod or None),
        view_kodu=kod or None,
        viewposition=g('ViewPosition'),          # bos bekleniyor, dogrulama icin
        rows=getattr(d, 'Rows', None), cols=getattr(d, 'Columns', None),
        photometric=g('PhotometricInterpretation'),
        bits=getattr(d, 'BitsStored', None),
        yas_dicom=g('PatientAge'),
        orient='|'.join(map(str, getattr(d, 'PatientOrientation', []) or [])) or None,
        spacing=g('ImagerPixelSpacing'),
        pencere=f"{g('WindowCenter')}/{g('WindowWidth')}",
        uretici=g('Manufacturer'),
        tetkik=g('StudyInstanceUID'), seri=g('SeriesInstanceUID'),
    )


kayit, hata = [], []
for i, p in enumerate(dicom_dosyalar):
    try:
        kayit.append(dicom_meta(p))
    except Exception as e:
        hata.append((os.path.relpath(p, DATA_ROOT), repr(e)))
    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(dicom_dosyalar)}')

meta = pd.DataFrame(kayit)
print(f'\nokunan: {len(meta)}   hatali: {len(hata)}')
for h in hata[:10]:
    print('  HATA', h)
meta.to_excel(f'{OZET_DIR}/dicom_meta.xlsx', index=False)
print('kaydedildi: dicom_meta.xlsx')
meta.head()

In [ ]:
print('cozunurluk:')
print((meta.rows.astype(str) + ' x ' + meta.cols.astype(str)).value_counts().to_string())

for k in ['view', 'view_kodu', 'taraf', 'photometric', 'bits', 'orient', 'spacing', 'pencere', 'uretici']:
    print(f'\n{k}:')
    print(meta[k].value_counts(dropna=False).head(8).to_string())

print('\nViewPosition dolu mu:', meta.viewposition.notna().sum(), '/', len(meta))
print('benzersiz hasta:', meta.hasta.nunique(),
      ' tetkik:', meta.tetkik.nunique(), ' seri:', meta.seri.nunique())

# Bit derinligi anomalisi varsa hangi dosyalar
if meta.bits.nunique() > 1:
    ana = meta.bits.mode()[0]
    print(f'\nBit derinligi anomalisi (cogunluk {ana}):')
    print(meta[meta.bits != ana][['yol', 'hasta', 'taraf', 'view', 'bits']].to_string(index=False))

# Yon bilgisi taraf+view'den turetilebiliyor mu (ek bilgi tasiyor mu)
print('\norient x (taraf, view):')
print(pd.crosstab(meta.orient, [meta.taraf, meta.view]).to_string())

## 4. Meme seviyesi yapi ve klinik etiketle eslesme

Meme = `(hasta, taraf)`. Meme seviyesi tahmin CC + MLO ortalamasi oldugu icin her memede iki
view'in bulunup bulunmadigi kritik. Ayrica **goruntulenmis ama klinik etiketi olmayan** meme
olup olmadigi burada ortaya cikiyor — etiketsiz meme egitime alinamaz.

In [ ]:
meta['meme'] = meta.hasta + '_' + meta.taraf
klinik['meme'] = klinik.ID1.astype(str) + '_' + klinik.LeftRight.astype(str)

print('hasta basina goruntu sayisi:')
print(meta.groupby('hasta').size().value_counts().sort_index().to_string())

print('\nmeme basina goruntu sayisi:')
print(meta.groupby('meme').size().value_counts().sort_index().to_string())
print('toplam goruntulenmis meme:', meta.meme.nunique())

print('\nview kombinasyonu (meme basina):')
komb = meta.groupby('meme')['view'].apply(lambda s: '+'.join(sorted(s.dropna().unique())))
print(komb.value_counts().to_string())
print('CC ve MLO birlikte olan meme:', int((komb == 'CC+MLO').sum()))

In [ ]:
gm, km = set(meta.meme), set(klinik.meme)
print('goruntulenmis meme:', len(gm))
print('klinik satir:', len(klinik), ' benzersiz klinik meme:', len(km),
      ' tekrarlanan:', int(klinik.meme.duplicated().sum()))
print('kesisim (etiketli + goruntulu):', len(gm & km))
print('goruntude olup KLINIKTE OLMAYAN meme:', len(gm - km), '  <-- egitime alinamaz')
print('klinikte olup goruntude olmayan:', len(km - gm), list(km - gm)[:5])

etiketsiz = meta[~meta.meme.isin(km)]
if len(etiketsiz):
    hasta_goruntu = meta.groupby('hasta').size()
    print(f'\netiketsiz goruntu: {len(etiketsiz)}  etiketsiz meme: {etiketsiz.meme.nunique()}'
          f'  etkilenen hasta: {etiketsiz.hasta.nunique()}')
    print('bu hastalarin toplam goruntu sayisi dagilimi:')
    print(hasta_goruntu[etiketsiz.hasta.unique()].value_counts().to_string())
    print('Yorum: iki memesi de goruntulenmis hastalarda yalnizca lezyonlu meme etiketli.')
    print('Etiketsiz kontralateral meme "saglikli" varsayilamaz (biyopsi dogrulamasi yok).')

print('\nhasta basina klinik satir:')
print(klinik.groupby('ID1').size().value_counts().to_string())
iki = klinik.groupby('ID1').filter(lambda g: len(g) > 1)
if len(iki):
    print(f'iki memesi de etiketli hasta: {iki.ID1.nunique()}')
    print('bu hastalarda iki memenin sinifi:')
    print(iki.groupby('ID1').classification.nunique().value_counts()
          .rename({1: 'ayni sinif', 2: 'karisik sinif'}).to_string())
    print('Not: karisik sinifli hastalar hasta-seviyesi stratified split i zorlastirir.')

## 5. Etiket dagilimi

Split ve dengeli ornekleme karari bu sayilara bagli. `D1-` / `D2-` on eki TCIA'ya gore iki farkli
alt kume; D2 neredeyse tamamen malign oldugu icin **karistirici (confounder)** olabilir.

In [ ]:
SINIF, ANORM, YAS, ALTTIP = 'classification', 'abnormality', 'Age', 'subtype'
etiketli = klinik[klinik.meme.isin(gm)].copy()
etiketli['grup'] = etiketli.ID1.astype(str).str[:2]

print('meme seviyesinde sinif dagilimi:')
print(etiketli[SINIF].value_counts().to_string())
n = etiketli[SINIF].value_counts()
print(f'oran: {n.min()/n.sum():.3f} / {n.max()/n.sum():.3f}')

print('\nalt kume (D1/D2) x sinif:')
print(pd.crosstab(etiketli.grup, etiketli[SINIF], margins=True).to_string())

print(f'\n{ANORM} x {SINIF}:')
print(pd.crosstab(etiketli[ANORM], etiketli[SINIF], margins=True).to_string())

print('\nyas:')
print(etiketli[YAS].describe().to_string())
print(etiketli.groupby(SINIF)[YAS].agg(['mean', 'std', 'min', 'max', 'count']).to_string())

print(f'\n{ALTTIP} doluluk: {etiketli[ALTTIP].notna().sum()}/{len(etiketli)}')
print(etiketli[ALTTIP].value_counts(dropna=False).to_string())
print('alt tip dolu olanlarin sinifi:')
print(pd.crosstab(etiketli[ALTTIP].notna(), etiketli[SINIF]).to_string())

print('\nnumber kolonu (meme basina goruntu sayisi olmali):')
print(etiketli['number'].value_counts(dropna=False).to_string())

## 6. Yinelenen piksel hash kontrolu

TCIA uyarisi: farkli hasta ID'lerinde ayni piksel hash'i bulunmus. Cakisanlar cikarilmazsa split
sizintisi olur. **Bu hucre 5202 goruntunun pikselini okur (~22 GB), uzun surer.**

In [ ]:
HASH_TARA = True  # kapatmak icin False

if HASH_TARA:
    hashler = {}
    for i, p in enumerate(dicom_dosyalar):
        try:
            a = pydicom.dcmread(p, force=True).pixel_array
            hashler[os.path.relpath(p, DATA_ROOT)] = hashlib.md5(np.ascontiguousarray(a)).hexdigest()
        except Exception as e:
            print('hash hatasi:', os.path.relpath(p, DATA_ROOT), repr(e))
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(dicom_dosyalar)}')

    hs = pd.Series(hashler, name='hash').rename_axis('yol').reset_index()
    hs = hs.merge(meta[['yol', 'hasta', 'taraf', 'view']], on='yol', how='left')
    hs.to_excel(f'{OZET_DIR}/piksel_hash.xlsx', index=False)

    grup = hs.groupby('hash')
    tekrar = grup.filter(lambda g: len(g) > 1)
    farkli_hasta = grup.filter(lambda g: g.hasta.nunique() > 1)
    print(f'\nbenzersiz hash: {hs.hash.nunique()} / {len(hs)} goruntu')
    print(f'tekrarlanan hash iceren goruntu: {len(tekrar)}')
    print(f'bunlardan FARKLI hastalara dagilmis: {len(farkli_hasta)}  <-- kritik')
    if len(farkli_hasta):
        print(farkli_hasta.sort_values('hash').to_string(index=False))
        print('\nEtkilenen hasta sayisi:', farkli_hasta.hasta.nunique())
else:
    farkli_hasta = pd.DataFrame()

### Dislama listesi

Cakisan hastalarin klinik etiketleri karsilastirilir. Etiketler celisiyorsa hangisinin dogru
oldugu bilinemez (TCIA de bilemiyor) -> ilgili hastalarin **tumu** cikarilir. Liste
`haric_hastalar.json` olarak yazilir, Parca 2 bunu okur.

In [ ]:
if HASH_TARA and len(farkli_hasta):
    haric_hastalar = sorted(farkli_hasta.hasta.unique())
    detay = []
    for h, grp in farkli_hasta.groupby('hash'):
        hl = sorted(grp.hasta.unique())
        et = klinik[klinik.ID1.isin(hl)][['ID1', 'LeftRight', 'Age', 'abnormality', 'classification']]
        print(f"hash {h[:12]} | {hl} | {sorted(set(grp.taraf + '-' + grp.view))}")
        print(et.to_string(index=False), '\n')
        detay.append(dict(hash=h[:12], hastalar=hl, goruntu=len(grp),
                          taraf_view=sorted(set(grp.taraf + '-' + grp.view))))

    with open(f'{OZET_DIR}/haric_hastalar.json', 'w', encoding='utf-8') as f:
        json.dump({'hastalar': haric_hastalar,
                   'gerekce': 'Ayni piksel verisi farkli hasta ID lerinde ve etiketler celisiyor.',
                   'detay': detay}, f, indent=2, ensure_ascii=False)

    kalan = klinik[(~klinik.ID1.isin(haric_hastalar)) & (klinik.meme.isin(gm))]
    print('haric hasta:', haric_hastalar)
    print(f'etiketli meme: {len(gm & km)} -> {len(kalan)}   goruntu: {2*len(gm & km)} -> {2*len(kalan)}')
    print('kalan sinif dagilimi:', kalan[SINIF].value_counts().to_dict())
    print('yazildi: haric_hastalar.json')
else:
    haric_hastalar = []

## 7. Piksel istatistikleri (ornek)

Ham piksel araligi ve arka plan orani — Parca 2'deki normalizasyon ve meme maskesi esigi icin.
Tum veriyi okumamak icin rastgele 40 goruntu.

In [ ]:
rng = np.random.default_rng(42)
ornek = rng.choice(len(dicom_dosyalar), size=min(40, len(dicom_dosyalar)), replace=False)

satir = []
for i in ornek:
    a = pydicom.dcmread(dicom_dosyalar[i], force=True).pixel_array
    satir.append(dict(yol=os.path.relpath(dicom_dosyalar[i], DATA_ROOT),
                      dtype=str(a.dtype), min=int(a.min()), max=int(a.max()),
                      ortalama=float(a.mean()),
                      sifir_orani=float((a == 0).mean()),
                      dusuk_orani=float((a < 10).mean())))
ps = pd.DataFrame(satir)
print(ps[['dtype', 'min', 'max', 'ortalama', 'sifir_orani', 'dusuk_orani']].describe().to_string())
print('\ndtype dagilimi:', ps.dtype.value_counts().to_dict())
print('arka plan (piksel<10) orani ortalamasi:', f'{ps.dusuk_orani.mean():.3f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ilk = pydicom.dcmread(dicom_dosyalar[ornek[0]], force=True).pixel_array
ax[0].imshow(ilk, cmap='gray'); ax[0].set_title('ornek goruntu'); ax[0].axis('off')
ax[1].hist(ilk.ravel(), bins=64, log=True); ax[1].set_title('piksel histogrami (log)')
ax[2].hist(ps.dusuk_orani, bins=15); ax[2].set_title('arka plan orani (40 ornek)')
plt.tight_layout(); plt.savefig(f'{OZET_DIR}/piksel_istatistik.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Grafikler

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))

a = ax[0, 0]
s = etiketli[SINIF].value_counts()
a.bar(s.index, s.values, color=['tab:red' if i.lower().startswith('mal') else 'tab:green' for i in s.index])
for i, v in enumerate(s.values):
    a.text(i, v, str(v), ha='center', va='bottom')
a.set_title('Sinif dagilimi (etiketli meme)'); a.set_ylabel('meme sayisi')

a = ax[0, 1]
ct = pd.crosstab(etiketli[ANORM], etiketli[SINIF])
ct.plot(kind='bar', ax=a, color=['tab:green', 'tab:red'], rot=0)
a.set_title('Anormallik tipi x sinif'); a.set_xlabel('')

a = ax[1, 0]
for etiket, grp in etiketli.groupby(SINIF):
    a.hist(grp[YAS].dropna(), bins=25, alpha=0.6, label=str(etiket))
a.legend(); a.set_xlabel('yas'); a.set_title('Yas dagilimi'); a.grid(alpha=0.3)

a = ax[1, 1]
ct2 = pd.crosstab(etiketli.grup, etiketli[SINIF])
ct2.plot(kind='bar', ax=a, color=['tab:green', 'tab:red'], rot=0)
a.set_title('Alt kume (D1/D2) x sinif  --  D2 karistirici mi?'); a.set_xlabel('')

plt.suptitle('CMMD2022 veri kesfi', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f'{OZET_DIR}/veri_kesfi.png', dpi=140, bbox_inches='tight')
plt.show()

## 9. Ozet — `PLAN.md` Bolum 1'e islenecek satirlar

In [ ]:
boyut = (meta.rows.astype(str) + 'x' + meta.cols.astype(str))
ozet = {
    'ortam': ORTAM,
    'data_root': DATA_ROOT,
    'toplam_dicom': len(meta),
    'okunamayan': len(hata),
    'hasta': int(meta.hasta.nunique()),
    'tetkik': int(meta.tetkik.nunique()),
    'goruntulenmis_meme': int(meta.meme.nunique()),
    'etiketli_meme': len(gm & km),
    'etiketsiz_meme': len(gm - km),
    'egitilebilir_goruntu': int(meta.meme.isin(km).sum()),
    'cozunurluk': f'{boyut.mode()[0]} ({boyut.nunique()} cesit)',
    'photometric': ', '.join(map(str, meta.photometric.dropna().unique())),
    'bits': meta.bits.value_counts().to_dict(),
    'piksel_spacing_mm': meta.spacing.mode()[0],
    'view_kaynagi': 'ViewCodeSequence (ViewPosition bos)',
    'view_dagilimi': meta.view.value_counts().to_dict(),
    'cc_mlo_tam_meme': int((komb == 'CC+MLO').sum()),
    'sinif_dagilimi': etiketli[SINIF].value_counts().to_dict(),
    'abnormality_dagilimi': etiketli[ANORM].value_counts().to_dict(),
    'D1_D2_x_sinif': pd.crosstab(etiketli.grup, etiketli[SINIF]).to_dict(),
    'iki_memesi_etiketli_hasta': int(iki.ID1.nunique()) if len(iki) else 0,
    'karisik_sinifli_hasta': int((iki.groupby('ID1').classification.nunique() == 2).sum()) if len(iki) else 0,
    'alttip_dolu': int(etiketli[ALTTIP].notna().sum()),
    'yinelenen_hash_farkli_hasta': len(farkli_hasta) if HASH_TARA else 'taranmadi',
    'haric_hasta': haric_hastalar,
    'kalan_etiketli_meme': len(gm & km) - int(klinik[klinik.ID1.isin(haric_hastalar)].meme.isin(gm).sum()),
    'arka_plan_orani': f'{ps.dusuk_orani.mean():.3f}',
}

for k, v in ozet.items():
    print(f'{k:28s}: {v}')

pd.DataFrame([{k: str(v) for k, v in ozet.items()}]).T.rename(columns={0: 'deger'}) \
    .to_excel(f'{OZET_DIR}/kesif_ozeti.xlsx')
print('\nkaydedildi:', f'{OZET_DIR}/kesif_ozeti.xlsx')